In [ ]:
# import os
# from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI

# load_dotenv()

# def load_llm(config):
#     provider = os.getenv("LLM_PROVIDER")

#     if provider == "openai":
#         return ChatOpenAI(
#             model=config["llm"]["openai_model"],
#             temperature=config["llm"]["temperature"]
#         )

#     elif provider == "gemini":
#         return ChatGoogleGenerativeAI(
#             model=config["llm"]["gemini_model"],
#             temperature=config["llm"]["temperature"]
#         )

#     else:
#         raise ValueError("Invalid LLM_PROVIDER")

In [4]:
print("hello")

hello


In [1]:
# --- Step 1: Load Libraries ---
import os
import json
import pandas as pd
from dotenv import load_dotenv

#for working with tabular data, loading files where rows and columns exists like csv, excels pandas 
#is used

# --- Step 2: Load .env ---
load_dotenv()

# --- Step 3: Load config.json ---
with open("config/config.json", "r") as f:
    config = json.load(f)
    
# --- Step 4: Load transcripts ---
df = pd.read_csv("data/transcripts.csv")

print("✅ Data Loaded")
print(df.head(), "\n")

# --- Step 5: Initialize LLM ---
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

#print("LLM_PROVIDER from env:", os.getenv("LLM_PROVIDER"))

def load_llm(config):
    provider = os.getenv("LLM_PROVIDER")

    if provider == "openai":
        llm = ChatOpenAI(
            model=config["llm"]["openai_model"],
            temperature=config["llm"]["temperature"]
        )

    elif provider == "gemini":
        llm = ChatGoogleGenerativeAI(
            model=config["llm"]["gemini_model"],
            temperature=config["llm"]["temperature"]
        )

    else:
        raise ValueError("Invalid LLM_PROVIDER")

    return llm

llm = load_llm(config)

print(f"✅ LLM Initialized using provider: {os.getenv('LLM_PROVIDER')}")

# --- Step 6: Test LLM ---
response = llm.invoke("Say 'setup successful' in one short sentence.")
print("\n🧠 LLM Response:")
print(response.content)

✅ Data Loaded
   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type  
0            billing  
1             claims  
2          complaint  
3      general_query   

✅ LLM Initialized using provider: openai

🧠 LLM Response:
Setup successful.


In [2]:
# --- Step 1: Imports ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
#Pydantic is a library for data validation and parsing. It enforces types at runtime, not 
# just during static analysis.
#Use case: When you’re dealing with external data (APIs, JSON, user input) and want automatic 
# validation + conversion.

#PydanticOutputParser LangChain ka ek output parser hai jo Pydantic models ke saath kaam karta hai.
# Jab LLM (Large Language Model) se response aata hai, woh ek string hota hai (jaise JSON text).
# PydanticOutputParser us string ko parse karke Pydantic model object bana deta hai.
# 👉 Matlab: LLM → JSON string → PydanticOutputParser → Python object with validation.

# --- Step 2: Define Output Schema ---
class ClassificationOutput(BaseModel):
    call_type: str = Field(description="Type of customer call")
    confidence: float = Field(description="Confidence score between 0 and 1")
    
parser = PydanticOutputParser(pydantic_object=ClassificationOutput)

# --- Step 3: Prompt Template ---
prompt = PromptTemplate(
    template="""
You are a call classification assistant.

Classify the following customer support transcript into one of these categories:
{labels}

Transcript:
{transcript}

{format_instructions}
""",
input_variables=["transcript"],
partial_variables={
        "format_instructions": parser.get_format_instructions(),
        "labels": config["classification"]["labels"]
    }
)

#{format_instructions} is passed internally by PydanticOutputParser

# --- Step 4: Create Chain ---
classification_chain = prompt | llm | parser

# --- Step 5: Test with one transcript ---
sample_text = df.iloc[0]["transcript"]
#iloc[0] is index location 0 means first row

result = classification_chain.invoke({
    "transcript": sample_text
})

print("✅ Classification Result:")
print(result) #call_type='billing' confidence=0.95

print("\n--------------------------------------------------\n")
print(json.dumps(result.model_dump(), indent=2))

✅ Classification Result:
call_type='billing' confidence=0.95

--------------------------------------------------

{
  "call_type": "billing",
  "confidence": 0.95
}


In [3]:
from tqdm import tqdm

results = []
for i, row in tqdm(df.iterrows(), total=len(df), desc="Classifying Calls"):
    try:
        output = classification_chain.invoke({
            "transcript": row["transcript"]
        })
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": output.call_type,
            "confidence": output.confidence
        })
    except Exception as e:
        print(f"❌ Error at row {i}: {e}")
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
        })

print(f"results: {results} \n")

'''
results: [{'call_id': 1, 'predicted_call_type': 'billing', 'confidence': 0.95}, 
          {'call_id': 2, 'predicted_call_type': 'claims', 'confidence': 0.95}, 
          {'call_id': 3, 'predicted_call_type': 'complaint', 'confidence': 0.9}, 
          {'call_id': 4, 'predicted_call_type': 'general_query', 'confidence': 0.9}]
'''

# Convert to DataFrame -> lists of dictionary into a proper table
results_df = pd.DataFrame(results)

# Merge with original data frame
df = df.merge(results_df, on="call_id")
print(df)

'''
df = df.drop(
    columns=[column for column in df.columns if column.startswith("predicted_call_type") or column.startswith("confidence")],
    errors="ignore"
)
df = df.merge(results_df, on="call_id", validate="one_to_one")

print(df)

The batch cell now:

Removes previous prediction columns before merging.
Handles repeated execution safely.
Validates one-to-one call_id matching.
Removes the obsolete failing batch cell.
'''

print("\n✅ Batch Classification Completed\n")
print(df[["call_id", "expected_call_type", "predicted_call_type", "confidence"]])

Classifying Calls: 100%|██████████| 4/4 [00:03<00:00,  1.10it/s]

results: [{'call_id': 1, 'predicted_call_type': 'billing', 'confidence': 0.95}, {'call_id': 2, 'predicted_call_type': 'claims', 'confidence': 0.95}, {'call_id': 3, 'predicted_call_type': 'complaint', 'confidence': 0.9}, {'call_id': 4, 'predicted_call_type': 'general_query', 'confidence': 0.9}] 

   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type predicted_call_type  confidence  
0            billing             billing        0.95  
1             claims              claims        0.95  
2          complaint           complaint        0.90  
3      general_query       general_query        0.90  

✅ Batch Classification Completed

   call_id e

In [4]:
# --- Step 1: Define routing logic ---

#on basis of knowledge_accuracy and resolution_quality we will evaluate billing call_type
def route_call(call_type):
    if call_type == "billing":
        return ["knowledge_accuracy", "resolution_quality"]
    
    elif call_type == "claims":
        return ["knowledge_accuracy", "resolution_quality"]
    
    elif call_type == "complaint":
        return ["tone_empathy", "resolution_quality"]
    
    elif call_type == "general_query":
        return ["knowledge_accuracy"]
    
    else:
        return ["knowledge_accuracy"]  # fallback


# --- Step 2: Apply routing to dataset ---

df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)
print(df) #evaluation_plan column is also added to data frame

print("\n✅ Routing Applied\n")
print(df[["call_id", "predicted_call_type", "evaluation_plan"]])

   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type predicted_call_type  confidence  \
0            billing             billing        0.95   
1             claims              claims        0.95   
2          complaint           complaint        0.90   
3      general_query       general_query        0.90   

                            evaluation_plan  
0  [knowledge_accuracy, resolution_quality]  
1  [knowledge_accuracy, resolution_quality]  
2        [tone_empathy, resolution_quality]  
3                      [knowledge_accuracy]  

✅ Routing Applied

   call_id predicted_call_type                           evaluation_plan
0        1     

In [5]:
# --- Step 1: Imports ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# --- Step 2: Define Output Schema ---
class ToneEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

tone_parser = PydanticOutputParser(pydantic_object=ToneEvaluation)

# --- Step 3: Prompt Template ---
tone_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the agent's tone and empathy in the following transcript.

Consider:
- Did the agent acknowledge the customer's issue?
- Was the tone polite and professional?
- Did the agent show empathy?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": tone_parser.get_format_instructions()
    }
)

# --- Step 4: Chain ---
tone_chain = tone_prompt | llm | tone_parser

# --- Step 5: Test on ONE complaint call (important)
sample_text = df[df["predicted_call_type"] == "complaint"].iloc[0]["transcript"]
print(f"\n sample_text: {sample_text} \n")

result = tone_chain.invoke({
    "transcript": sample_text
})

print("✅ Tone Evaluation Result:")
print(result)


 sample_text: Customer: This is the third time I'm calling! Nothing is resolved. Agent: I apologize for the inconvenience... 

✅ Tone Evaluation Result:
score=3 reasoning="The agent acknowledged the customer's issue by apologizing for the inconvenience, which indicates some level of recognition of the customer's frustration. However, the response lacks depth in empathy and does not fully address the customer's feelings or the repeated nature of their calls. The tone is polite and professional, but it could benefit from a more empathetic approach to better connect with the customer."


In [6]:
# --- Step 1: Schema ---
class ResolutionEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

resolution_parser = PydanticOutputParser(pydantic_object=ResolutionEvaluation)

# --- Step 2: Prompt ---
resolution_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the resolution quality of the agent.

Consider:
- Did the agent fully resolve the customer's issue?
- Were next steps clearly communicated?
- Did the agent confirm resolution before ending?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": resolution_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
resolution_chain = resolution_prompt | llm | resolution_parser

# --- Step 4: Test
sample_text = df.iloc[0]["transcript"]

result = resolution_chain.invoke({
    "transcript": sample_text
})

print("✅ Resolution Evaluation Result:")
print(result)

✅ Resolution Evaluation Result:
score=2 reasoning="The agent did not fully resolve the customer's issue as the transcript ends abruptly without confirming the resolution or providing next steps. There is no indication that the agent addressed the double charge or offered a solution."


In [7]:
# --- Step 1: Imports (safe to repeat) ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# --- Step 2: Schema ---
class KnowledgeEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

knowledge_parser = PydanticOutputParser(pydantic_object=KnowledgeEvaluation)

# --- Step 3: Prompt ---
knowledge_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the agent's knowledge accuracy and clarity.

Consider:
- Did the agent provide correct and relevant information?
- Was the explanation clear and easy to understand?
- Did the agent avoid vague or misleading statements?

IMPORTANT:
- If the transcript does not contain enough information, give a moderate score (2 or 3) and explain why.

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": knowledge_parser.get_format_instructions()
    }
)

# --- Step 4: Chain ---
knowledge_chain = knowledge_prompt | llm | knowledge_parser

# --- Step 4: Test
sample_text = df.iloc[0]["transcript"]

result = resolution_chain.invoke({
    "transcript": sample_text
})

print("✅ Knowledge Evaluation Result:")
print(result)

✅ Knowledge Evaluation Result:
score=2 reasoning="The agent did not fully resolve the customer's issue as the transcript ends abruptly without confirming the resolution or providing next steps. There is no indication that the agent addressed the double charge."


In [8]:
from tqdm import tqdm

# --- Step 1: Evaluation Runner ---

def run_evaluations(transcript, eval_plan):
    results = {}
    #where results is empty dictionary so results["tone"] is key and tone_result.model_dump() is value
    #so key value pair
    if "tone_empathy" in eval_plan:
        try:
            tone_result = tone_chain.invoke({"transcript": transcript})
            results["tone"] = tone_result.model_dump()
        except Exception as e:
            results["tone"] = {"error": str(e)}

    if "knowledge_accuracy" in eval_plan:
        try:
            knowledge_result = knowledge_chain.invoke({"transcript": transcript})
            results["knowledge"] = knowledge_result.model_dump()
        except Exception as e:
            results["knowledge"] = {"error": str(e)}

    if "resolution_quality" in eval_plan:
        try:
            resolution_result = resolution_chain.invoke({"transcript": transcript})
            results["resolution"] = resolution_result.model_dump()
        except Exception as e:
            results["resolution"] = {"error": str(e)}

    return results


# --- Step 2: Apply to entire dataset ---

evaluation_outputs = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Running Evaluations"):
    output = run_evaluations(row["transcript"], row["evaluation_plan"])
    
    evaluation_outputs.append({
        "call_id": row["call_id"],
        "evaluation_output": output
    })

# Convert to DataFrame
eval_df = pd.DataFrame(evaluation_outputs)

# Merge
df = df.merge(eval_df, on="call_id")

print("\n✅ Evaluation Completed\n")

# Show one example clearly
import pprint
pprint.pprint(df.iloc[0]["evaluation_output"])

Running Evaluations: 100%|██████████| 4/4 [00:09<00:00,  2.47s/it]


✅ Evaluation Completed

{'knowledge': {'reasoning': 'The transcript does not provide enough '
                            "information about the agent's response after "
                            "checking the customer's issue. While the agent's "
                            'initial acknowledgment of the problem is '
                            'appropriate, there is no follow-up information or '
                            'resolution provided, making it difficult to fully '
                            "evaluate the accuracy and clarity of the agent's "
                            'knowledge.',
               'score': 3},
 'resolution': {'reasoning': "The agent did not fully resolve the customer's "
                             'issue as the transcript ends abruptly without '
                             'confirming the resolution or providing next '
                             'steps. There is no indication that the agent '
                             'addressed the double char

In [9]:
# --- Step 1: Schema ---
class FinalReport(BaseModel):
    summary: str = Field(description="Overall evaluation summary")
    recommendations: list[str] = Field(description="List of actionable improvements")

final_parser = PydanticOutputParser(pydantic_object=FinalReport)

# --- Step 2: Prompt ---
final_prompt = PromptTemplate(
    template="""
You are a QA manager reviewing customer support calls.

Based on the evaluation results below, generate:

1. A concise summary of the agent's performance
2. A list of actionable recommendations for improvement

Evaluation Data:
{evaluation_output}

IMPORTANT:
- Be specific and practical
- Do not repeat scores
- Focus on improvement

{format_instructions}
""",
    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
final_chain = final_prompt | llm | final_parser

# --- Step 4: Test on one row
sample_eval = df.iloc[0]["evaluation_output"]

result = final_chain.invoke({
    "evaluation_output": sample_eval
})

print("✅ Final QA Report:")
print(result)

✅ Final QA Report:
summary="The agent demonstrated initial acknowledgment of the customer's issue but failed to provide a resolution or follow-up information. This lack of clarity and closure negatively impacted the overall effectiveness of the support interaction." recommendations=['Ensure that all customer issues are fully addressed by providing clear resolutions and next steps before ending the call.', 'Enhance knowledge of common issues and solutions to improve the ability to provide accurate and helpful information during calls.', 'Practice active listening techniques to better understand customer concerns and respond appropriately.', 'Implement a checklist for closing calls to confirm that all customer questions have been answered and issues resolved.']


In [10]:
from tqdm import tqdm

final_outputs = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Generating Final Reports"):
    try:
        result = final_chain.invoke({
            "evaluation_output": row["evaluation_output"]
        })

        final_outputs.append({
            "call_id": row["call_id"],
            "summary": result.summary,
            "recommendations": result.recommendations
        })

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")
        final_outputs.append({
            "call_id": row["call_id"],
            "summary": None,
            "recommendations": None
        })

# Convert to DataFrame
final_df = pd.DataFrame(final_outputs)

# Merge
df = df.merge(final_df, on="call_id")

print("\n✅ Final Reports Generated\n")

# Show final result
df[[
    "call_id",
    "predicted_call_type",
    "evaluation_output",
    "summary",
    "recommendations"
]]

Generating Final Reports: 100%|██████████| 4/4 [00:08<00:00,  2.16s/it]


✅ Final Reports Generated



,call_id,predicted_call_type,evaluation_output,summary,recommendations
0,1,billing,"{'knowledge': {'score': 3, 'reasoning': 'The t...",The agent demonstrated an initial understandin...,[Ensure to provide a clear resolution or next ...
1,2,claims,"{'knowledge': {'score': 2, 'reasoning': 'The t...",The agent demonstrated insufficient knowledge ...,[Enhance product and claim knowledge through r...
2,3,complaint,"{'tone': {'score': 3, 'reasoning': 'The agent ...",The agent demonstrated a polite and profession...,[Enhance empathy by using more personalized la...
3,4,general_query,"{'knowledge': {'score': 2, 'reasoning': 'The t...",The agent demonstrated insufficient knowledge ...,[Enhance product knowledge training to ensure ...


In [12]:
df.to_excel("data/output.xlsx", index=False)
print("✅ Results saved to data/output.xlsx")

✅ Results saved to data/output.xlsx
